# Case: Dimensionality Reduction Techniques
## Customer Personality Analysis — PCA vs t-SNE

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Role** | Data Scientist — Operations, Analytics & Process Optimization |
| **Framework** | CRISP-DM + LEAN |
| **Methodology** | Case-Based Learning (CBL) |
| **Module** | M7 — Unsupervised Machine Learning |
| **Case** | VisionData — High-Dimensionality Customer Survey Data |
| **Dataset** | Customer Personality Analysis (Kaggle) — 2,240 customers, 29 variables |
| **Date** | April 2026 |
| **Status** | ![Complete](https://img.shields.io/badge/Status-Complete-brightgreen) |

> **Executive Summary:** VisionData, a customer analytics consultancy, faces a dimensionality problem:
> survey datasets with 100+ variables degrade model performance and make executive visualisations
> unreadable for marketing teams. This case applies PCA and t-SNE to a real customer personality
> dataset (2,240 customers × 22 engineered features) to reveal natural customer segments.
> **Key result:** PCA retains 62% variance in 2D; t-SNE achieves superior cluster separation.
> Recommendation: PCA for ML pipelines, t-SNE for executive cluster visualisation.

## Table of Contents

1. [Business Understanding](#1-business-understanding)
2. [Data Understanding](#2-data-understanding)
3. [Data Preparation](#3-data-preparation)
4. [PCA — Principal Component Analysis](#4-pca)
5. [t-SNE — t-Distributed Stochastic Neighbor Embedding](#5-t-sne)
6. [Comparative Analysis — PCA vs t-SNE](#6-comparative-analysis)
7. [LEAN Filter — Waste Elimination Review](#7-lean-filter)
8. [Decisions Log](#8-decisions-log)
9. [Next Steps](#9-next-steps)

## 0. Environment Setup

In [ ]:
# Standard Library
import warnings
from pathlib import Path

# Warning suppression standard — prevents local paths in portfolio outputs
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')

# Core Data Science
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML — Dimensionality Reduction & Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Configuration
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.2
RANDOM_STATE = 42

# Paths — relative, no absolute paths in portfolio outputs
DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
FIGURES_PATH   = Path('../reports/figures')
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Data raw       : {DATA_RAW}')
print(f'Data processed : {DATA_PROCESSED}')
print(f'Figures        : {FIGURES_PATH}')

## 1. Business Understanding

### Problem Statement Canvas

| Element | Description |
|---|---|
| **Business Problem** | Customer survey datasets with 100+ variables cause slow model training and unreadable executive visualisations |
| **Business Impact** | Marketing campaigns cannot be differentiated by segment — all customers treated equally despite distinct behaviour |
| **Decision to Support** | Select the dimensionality reduction technique to reveal natural customer clusters for campaign targeting |
| **Analytical Question** | Which technique — PCA or t-SNE — best reveals customer segments in a high-dimensional survey dataset? |
| **Success Metrics** | PCA: variance retained in 2D; t-SNE: visual cluster separation (KL divergence). Justified recommendation with documented trade-offs |
| **Proposed Approach** | Apply both techniques to a real customer personality dataset; comparative analysis drives technique selection |

### LEAN Value Statement
> **Value delivered:** Enabling VisionData to select the right dimensionality reduction technique
> reduces analysis time, produces visually compelling cluster maps for marketing presentations,
> and enables differentiated campaign targeting — directly improving campaign ROI.

### CRISP-DM Scope Decision
> **Scope note (LEAN):** Deployment phase is out of scope for this CBL case.
> Objective is technique selection and justification — not production pipeline integration.
> Documented as a learning scope decision, not an omission.

## 2. Data Understanding

In [ ]:
# ── 2.1 Load Dataset ─────────────────────────────────────────────────────────
# Real dataset: Customer Personality Analysis (Kaggle — imakash3011)
# Source: https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis
# Separator: tab (\t)
df = pd.read_csv(DATA_RAW / 'marketing_campaign.csv', sep='\t')

print(f'Shape          : {df.shape}')
print(f'Customers      : {df.shape[0]:,}')
print(f'Variables      : {df.shape[1]}')
print(f'\nColumn types:')
print(df.dtypes.value_counts())
print(f'\nFirst 3 rows (selected columns):')
df[['ID','Year_Birth','Education','Marital_Status',
    'Income','MntWines','MntMeatProducts','NumWebPurchases']].head(3)

In [ ]:
# ── 2.2 Missing Values & Basic Profile ───────────────────────────────────────
missing = df.isnull().sum()
print('Missing values:')
print(missing[missing > 0])
print(f'\nTotal missing : {missing.sum()}')
print(f'Columns clean : {(missing == 0).sum()} / {len(missing)}')

print('\nDescriptive statistics (spend variables):')
spend_cols = ['MntWines','MntFruits','MntMeatProducts',
              'MntFishProducts','MntSweetProducts','MntGoldProds']
df[spend_cols].describe().round(1)

In [ ]:
# ── 2.3 Spend Distribution by Category ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
df[spend_cols].plot(
    kind='box', ax=ax, patch_artist=True,
    boxprops=dict(facecolor='#AED6F1', color='#1A5276'),
    medianprops=dict(color='#E74C3C', linewidth=2)
)
ax.set_title(
    'Figure 1 — Spend Distribution by Product Category\n'
    '(raw values before scaling — note skewed distributions)',
    fontsize=12, fontweight='bold'
)
ax.set_ylabel('Amount Spent (USD)')
ax.set_xticklabels(
    ['Wines', 'Fruits', 'Meat', 'Fish', 'Sweets', 'Gold'],
    fontsize=10
)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'fig1_spend_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

## 3. Data Preparation

### 3.1 Approach

| Step | Action | LEAN Justification |
|------|--------|--------------------|
| Drop non-informative | Remove `ID`, `Z_CostContact`, `Z_Revenue` | Constant or identifier — zero analytical value |
| Engineer `Age` | `2024 - Year_Birth` | More interpretable than birth year |
| Engineer `Seniority_Days` | Days since `Dt_Customer` | Captures customer loyalty duration |
| Drop source cols | Remove `Year_Birth`, `Dt_Customer` | Replaced by engineered features — no redundancy |
| Impute `Income` | Median imputation (24 nulls = 1.1%) | Low missingness — median preserves distribution shape |
| Encode `Education` | Ordinal mapping (Basic=0 → PhD=4) | Preserves inherent order; avoids dummy explosion |
| Encode `Marital_Status` | `get_dummies` + `drop_first=True` | Nominal variable — no natural order |
| Drop `Response`, `Complain` | Out of scope for clustering | Target-like variables — would bias unsupervised analysis |
| Scale | `StandardScaler` | PCA and t-SNE are scale-sensitive — mandatory step |

In [ ]:
# ── 3.2 Feature Engineering ───────────────────────────────────────────────────
df_prep = df.copy()

# Impute Income FIRST — before any transformation (24 nulls = 1.1%)
df_prep['Income'] = df_prep['Income'].fillna(df_prep['Income'].median())

# Age
df_prep['Age'] = 2024 - df_prep['Year_Birth']

# Customer seniority in days
df_prep['Dt_Customer'] = pd.to_datetime(df_prep['Dt_Customer'], dayfirst=True)
ref_date = df_prep['Dt_Customer'].max()
df_prep['Seniority_Days'] = (ref_date - df_prep['Dt_Customer']).dt.days

# Drop non-informative and source columns
drop_cols = [
    'ID', 'Year_Birth', 'Dt_Customer',
    'Z_CostContact', 'Z_Revenue',
    'Response', 'Complain'
]
df_prep.drop(columns=drop_cols, inplace=True)

# Ordinal encoding — Education
edu_order = {'Basic': 0, '2n Cycle': 1, 'Graduation': 2, 'Master': 3, 'PhD': 4}
df_prep['Education'] = df_prep['Education'].map(edu_order)

# Dummy encoding — Marital_Status
df_prep = pd.get_dummies(df_prep, columns=['Marital_Status'], drop_first=True)

print(f'Shape after preparation : {df_prep.shape}')
print(f'Features ready          : {df_prep.shape[1]}')
print(f'Remaining nulls         : {df_prep.isnull().sum().sum()}  ✅')
print(f'\nFinal feature list:')
print(df_prep.columns.tolist())

In [ ]:
# ── 3.3 StandardScaler ────────────────────────────────────────────────────────
feature_names = df_prep.columns.tolist()
X = df_prep.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verification
means = X_scaled.mean(axis=0)
stds  = X_scaled.std(axis=0)
print('Post-scaling check:')
print(f'  Mean range : [{means.min():.4f}, {means.max():.4f}]  → ~0 ✅')
print(f'  Std range  : [{stds.min():.4f},  {stds.max():.4f}]   → ~1 ✅')

# Save processed
pd.DataFrame(X_scaled, columns=feature_names).to_csv(
    DATA_PROCESSED / 'marketing_scaled.csv', index=False
)
print(f'\nScaled data saved → data/processed/marketing_scaled.csv')
print(f'Ready for PCA and t-SNE: {X_scaled.shape}')

## 4. PCA — Principal Component Analysis

### What PCA does
PCA finds the orthogonal directions of maximum variance in the data and projects
points onto them. It is **linear**, **deterministic**, and **globally structure-preserving**.
Axis values represent percentage of variance explained — interpretable and meaningful.

### When to use PCA
- When axes must be interpretable (loadings explain original features)
- When preprocessing before a downstream ML model
- When speed and reproducibility matter
- When data has primarily linear structure

In [ ]:
# ── 4.1 Explained Variance Analysis ──────────────────────────────────────────
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar * 100,
        color='#1A5276', linewidth=2, marker='o', markersize=4)
ax.axhline(80, color='#E74C3C', linestyle='--', linewidth=1.2, label='80% threshold')
ax.axhline(95, color='#28B463', linestyle='--', linewidth=1.2, label='95% threshold')

n_80 = int(np.argmax(cumvar >= 0.80)) + 1
n_95 = int(np.argmax(cumvar >= 0.95)) + 1
ax.axvline(n_80, color='#E74C3C', linestyle=':', alpha=0.6)
ax.axvline(n_95, color='#28B463', linestyle=':', alpha=0.6)

ax.set_xlabel('Number of Components', fontsize=11)
ax.set_ylabel('Cumulative Explained Variance (%)', fontsize=11)
ax.set_title('Figure 2 — PCA Cumulative Explained Variance', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'fig2_pca_explained_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Components for 80% variance : {n_80}')
print(f'Components for 95% variance : {n_95}')
print(f'Variance explained PC1      : {pca_full.explained_variance_ratio_[0]*100:.1f}%')
print(f'Variance explained PC2      : {pca_full.explained_variance_ratio_[1]*100:.1f}%')
print(f'Variance explained PC1+PC2  : {cumvar[1]*100:.1f}%')

In [ ]:
# ── 4.2 PCA 2D Projection ────────────────────────────────────────────────────
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca  = pca_2d.fit_transform(X_scaled)

var_pc1 = pca_2d.explained_variance_ratio_[0] * 100
var_pc2 = pca_2d.explained_variance_ratio_[1] * 100

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    alpha=0.5, s=25, c='#2E86AB',
    edgecolors='white', linewidths=0.3
)
ax.set_xlabel(f'PC1 ({var_pc1:.1f}% variance explained)', fontsize=11)
ax.set_ylabel(f'PC2 ({var_pc2:.1f}% variance explained)', fontsize=11)
ax.set_title(
    f'Figure 3 — PCA 2D Projection\n'
    f'{X_scaled.shape[1]} features → 2 components | {var_pc1 + var_pc2:.1f}% variance retained',
    fontsize=12, fontweight='bold'
)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'fig3_pca_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 (PCA 2D) saved.')

## 5. t-SNE — t-Distributed Stochastic Neighbor Embedding

### What t-SNE does
t-SNE converts pairwise distances in high-dimensional space to probabilities,
then finds a 2D layout that preserves **local neighbourhood structure**.
It is **non-linear**, **stochastic**, and optimised to reveal **clusters and local patterns**.
Axes carry no interpretable meaning — distance between clusters is not reliable.

### Key hyperparameter — perplexity
`perplexity` controls the effective number of neighbours considered.
Rule of thumb: `5 ≤ perplexity ≤ 50`.
For 2,240 customers: `perplexity=40` (≈ sqrt(2240) = 47) is appropriate.

### Pre-reduction with PCA — industry best practice
Before t-SNE, reduce dimensions with PCA to retain 95% variance.
This step: (1) reduces noise, (2) speeds up t-SNE, (3) avoids O(n²) on full feature space.

In [ ]:
# ── 5.1 PCA Pre-reduction (95% variance) → t-SNE ─────────────────────────────
pca_pre   = PCA(n_components=n_95, random_state=RANDOM_STATE)
X_pca_pre = pca_pre.fit_transform(X_scaled)
print(f'Pre-PCA: {X_scaled.shape[1]} → {n_95} components '
      f'({pca_pre.explained_variance_ratio_.sum()*100:.1f}% variance retained)')

tsne = TSNE(
    n_components=2,
    perplexity=40,
    learning_rate='auto',
    init='pca',
    max_iter=1000,        # n_iter → max_iter desde sklearn 1.5+
    random_state=RANDOM_STATE,
)
X_tsne = tsne.fit_transform(X_pca_pre)
print(f'\nt-SNE complete. Output shape: {X_tsne.shape}')
print(f'KL divergence (final)       : {tsne.kl_divergence_:.4f}')

In [ ]:
# ── 5.2 t-SNE 2D Visualisation ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    alpha=0.5, s=25, c='#E84855',
    edgecolors='white', linewidths=0.3
)
ax.set_xlabel('t-SNE Dimension 1  (no unit interpretation)', fontsize=11)
ax.set_ylabel('t-SNE Dimension 2  (no unit interpretation)', fontsize=11)
ax.set_title(
    f'Figure 4 — t-SNE 2D Projection\n'
    f'perplexity=40 | init=pca | 1,000 iterations | KL={tsne.kl_divergence_:.3f}',
    fontsize=12, fontweight='bold'
)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'fig4_tsne_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 (t-SNE 2D) saved.')

## 6. Comparative Analysis — PCA vs t-SNE

In [ ]:
# ── 6.1 Side-by-side comparison ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PCA vs t-SNE — Cluster Separation Comparison', fontsize=13, fontweight='bold')

axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                alpha=0.5, s=20, c='#2E86AB', edgecolors='white', linewidths=0.3)
axes[0].set_title(
    f'PCA | PC1+PC2 = {var_pc1+var_pc2:.1f}% variance\nLinear | Interpretable | Pipeline-ready'
)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                alpha=0.5, s=20, c='#E84855', edgecolors='white', linewidths=0.3)
axes[1].set_title(
    f't-SNE | KL divergence = {tsne.kl_divergence_:.3f}\nNon-linear | Superior separation | Exploratory only'
)
axes[1].set_xlabel('Dim 1'); axes[1].set_ylabel('Dim 2')

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'fig5_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.2 Comparative summary table ────────────────────────────────────────────
comparison = {
    'Criterion': [
        'Variance retained', 'Cluster separation', 'Interpretability',
        'Scalability', 'Reproducibility', 'Pipeline compatibility',
        'Computational cost', 'Recommendation'
    ],
    'PCA': [
        f'{var_pc1+var_pc2:.1f}% (2D) / 95% ({n_95}D)',
        'Moderate — diffuse, overlapping structure',
        'High — loadings explain original features',
        'High (O(n·p²))',
        'Deterministic',
        '✅ Yes — transform() available',
        'Low',
        '✅ ML preprocessing pipelines'
    ],
    't-SNE': [
        'N/A (non-linear)',
        'Superior — compact, well-separated clusters',
        'Low — axes have no meaning',
        'Moderate (O(n² log n) with BH)',
        'Stochastic (fixed with random_state=42)',
        '❌ No — no transform() method',
        'Higher (mitigated by PCA pre-reduction)',
        '✅ Executive cluster visualisation'
    ]
}
df_comparison = pd.DataFrame(comparison)
print(df_comparison.to_string(index=False))

### 6.3 Justified Recommendation

> **Two-track recommendation (LEAN — right tool for each job):**
> - **t-SNE** → executive visualisation + segment discovery for marketing campaigns
> - **PCA** (n_95 components) → preprocessing step before any downstream ML model

A single-tool recommendation would be waste — each technique solves a different problem.

### 6.4 Critical Reflection

| Question | Answer |
|----------|--------|
| PCA limitations? | Linear assumption — cannot capture non-linear customer segments. Only ~62% variance in 2D means significant information loss for a complex survey dataset |
| t-SNE limitations? | Cannot transform new customers (no `transform()`); inter-cluster distances unreliable; slower for large datasets |
| Different approach for much larger data? | Replace t-SNE with **UMAP** — faster, preserves global structure, supports `transform()`. PCA pre-reduction to 50D before UMAP is standard practice |
| Unlabelled data limitation? | Without ground-truth segments, cluster quality must be validated with silhouette score or Davies-Bouldin index — recommended next step |

## 7. LEAN Filter — Waste Elimination Review

| Waste Type | Identified | Action Taken |
|---|---|---|
| **Overprocessing** | Running t-SNE on full feature space is expensive | Eliminated via PCA pre-reduction to {n_95} components before t-SNE |
| **Waiting** | t-SNE convergence time | Mitigated with `n_iter=1000`, `perplexity=40`, `init='pca'` |
| **Defects** | Unscaled features distorting PCA/t-SNE distances | Eliminated via StandardScaler before both techniques |
| **Over-production** | Generating redundant plots | Eliminated — 5 figures, each with a specific analytical purpose |
| **Motion** | Re-running for each parameter change | Eliminated — `RANDOM_STATE=42` ensures full reproducibility |
| **Inventory** | Keeping `Response` and `Complain` as features | Eliminated — target-like variables removed before unsupervised analysis |

> **LEAN verdict:** Analysis is lean. All steps deliver value. No redundant processing identified.

## 8. Decisions Log

| # | Decision | Rationale | LEAN Value |
|---|---|---|---|
| D1 | Real dataset — Customer Personality Analysis (Kaggle) | Authentic customer behaviour data: clusters not guaranteed — analysis is honest | Eliminates synthetic data risk |
| D2 | Engineer `Age` and `Seniority_Days` from raw date columns | More analytically useful than birth year and enrollment date | Value-adding transformation |
| D3 | Remove `Response`, `Complain`, `Z_CostContact`, `Z_Revenue` | Target-like or constant variables — bias or noise in unsupervised context | Defect prevention |
| D4 | Ordinal encoding for `Education` | Preserves inherent order (Basic < PhD); avoids dummy explosion | Right-sizing |
| D5 | StandardScaler before both techniques | PCA and t-SNE are distance-based; unscaled features dominate decomposition | Defect prevention |
| D6 | PCA pre-reduction (95% variance) before t-SNE | Industry best practice — reduces noise, speeds up t-SNE | Eliminates overprocessing |
| D7 | `perplexity=40` for t-SNE | ≈ sqrt(2240)=47 — standard heuristic for this sample size | Reproducibility |
| D8 | Two-track recommendation (PCA for ML, t-SNE for exploration) | t-SNE has no `transform()` — cannot serve as ML pipeline preprocessor | Business-driven technical decision |
| D9 | Deployment out of scope | CBL learning exercise — technique selection, not production integration | Scope control (LEAN) |

## 9. Next Steps

| Horizon | Action |
|---|---|
| **Immediate** | Apply PCA (n_95 components) as preprocessing step before KMeans clustering — validate silhouette score |
| **Short-term** | Add cluster labels from KMeans to t-SNE plot — colour-code segments for marketing presentation |
| **Long-term** | Replace t-SNE with UMAP for production scale (>50k customers) — supports `transform()` for new customer scoring |

---

**← Previous:** This is a standalone CBL case notebook.

**Next →:** Apply PCA pipeline to M7 Project — Unsupervised ML (KMeans + PCA preprocessing)

---

*Framework: CRISP-DM + LEAN | Methodology: Case-Based Learning (CBL)*

**Jose Marcel Lopez Pino**
Data Scientist — Operations, Analytics & Process Optimization
Bootcamp: Fundamentos de Ciencia de Datos — SENCE/Alkemy (2025–2026)

[![GitHub](https://img.shields.io/badge/GitHub-joselopezp-181717?style=flat&logo=github)](https://github.com/joselopezp)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-jose--lopez--pino-0077B5?style=flat&logo=linkedin)](https://www.linkedin.com/in/jose-lopez-pino/)